In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve
)

# Import wrapper module so joblib can deserialize threshold-aware models.
from thresholding import ThresholdedClassifier


# Load test data
X_test = pd.read_csv(Path(r"data/X_test_selected.csv"))
y_test = pd.read_csv(Path(r"data/y_test.csv")).squeeze("columns")

# Load trained threshold-aware models
logreg_model = joblib.load(Path(r"models/logreg_model.joblib"))
rf_model = joblib.load(Path(r"models/rf_model.joblib"))
xgb_model = joblib.load(Path(r"models/xgb_model.joblib"))
cat_boost = joblib.load(Path(r"models/cat_boost.joblib"))

models = {
    "Logistic Regression": logreg_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "CatBoost": cat_boost,
}

print("Data and threshold-aware models loaded.")
for model_name, model in models.items():
    model_threshold = getattr(model, "threshold", 0.5)
    print(f"{model_name} threshold: {model_threshold:.2f}")


## Apply frozen operating policy from validation on full test set


In [ ]:
policy_path = Path(r"models/model_operating_points.csv")

if not policy_path.exists():
    raise FileNotFoundError(
        "Missing models/model_operating_points.csv. Run Modelling.ipynb first to freeze policy on validation."
    )

frozen_policy_df = pd.read_csv(policy_path)
required_cols = {"model", "selected_selection_rate", "derived_threshold"}
missing_cols = required_cols - set(frozen_policy_df.columns)
if missing_cols:
    raise ValueError(f"Operating policy file missing columns: {missing_cols}")

# Evaluate with base estimators even if wrappers are loaded.
models_for_eval = {name: getattr(model, "base_model", model) for name, model in models.items()}

probability_dict = {}
predictions_dict = {}
final_metrics_rows = []

for model_name, model in models_for_eval.items():
    policy_row = frozen_policy_df[frozen_policy_df["model"] == model_name]
    if policy_row.empty:
        raise ValueError(f"No frozen policy found for model: {model_name}")

    threshold = float(policy_row["derived_threshold"].iloc[0])
    selected_rate = float(policy_row["selected_selection_rate"].iloc[0])

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    probability_dict[model_name] = y_prob
    predictions_dict[model_name] = {
        "y_pred": y_pred,
        "y_prob": y_prob,
        "threshold": threshold,
        "selection_rate": selected_rate,
    }

    prevalence = y_test.mean()
    precision = precision_score(y_test, y_pred, zero_division=0)

    final_metrics_rows.append(
        {
            "model": model_name,
            "frozen_selection_rate_from_validation": selected_rate,
            "frozen_threshold_from_validation": threshold,
            "test_realized_selection_rate": y_pred.mean(),
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision,
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_prob),
            "lift_vs_random": (precision / prevalence) if prevalence > 0 else np.nan,
        }
    )

final_metrics_df = pd.DataFrame(final_metrics_rows).sort_values("roc_auc", ascending=False)

print("Frozen policy loaded (chosen on validation only):")
display(frozen_policy_df)

print("Final unbiased test metrics using frozen thresholds:")
display(final_metrics_df)


In [ ]:
# Optional operational ranking table for business use (no threshold tuning)
ranking_rows = []
for model_name, y_prob in probability_dict.items():
    ranking_rows.append(
        pd.DataFrame(
            {
                "model": model_name,
                "row_id": np.arange(len(y_prob)),
                "y_true": y_test.values,
                "score": y_prob,
            }
        )
        .sort_values("score", ascending=False)
        .head(20)
    )

ranking_preview_df = pd.concat(ranking_rows, ignore_index=True)
print("Top 20 highest-probability clients per model (test set preview):")
display(ranking_preview_df)


In [ ]:
policy_vs_test_df = final_metrics_df.merge(
    frozen_policy_df[["model", "selected_selection_rate", "derived_threshold"]],
    on="model",
    how="left",
)

display_cols = [
    "model",
    "selected_selection_rate",
    "derived_threshold",
    "test_realized_selection_rate",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "lift_vs_random",
]

print("Frozen policy vs test outcomes:")
display(policy_vs_test_df[display_cols].sort_values("roc_auc", ascending=False))


## Classification reports e confusion matrixes finais (threshold congelado da valida??o)


In [ ]:
for model_name, preds in predictions_dict.items():
    print(f"\n{'='*60}")
    print(model_name)
    print(f"{'='*60}")
    print(
        f"Frozen selection rate (validation): {preds['selection_rate']:.2%} | "
        f"Frozen threshold (validation): {preds['threshold']:.4f}"
    )

    cm = confusion_matrix(y_test, preds["y_pred"])

    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)

    disp.plot(
        ax=ax,
        cmap="Oranges",
        colorbar=False,
        values_format="d"
    )

    plt.title(f"{model_name} - Confusion Matrix")
    plt.show()

    print("\nClassification Report:")
    print(classification_report(y_test, preds["y_pred"], zero_division=0))


## ROC Curve on full test set (probability ranking quality)


In [ ]:
plt.figure(figsize=(8, 6))

for model_name, y_prob in probability_dict.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve on Test Set")
plt.legend()
plt.show()


## Final feature importances para Random Forest e XGboost


In [ ]:
feature_names = X_test.columns

rf_base = getattr(rf_model, "base_model", rf_model)
xgb_base = getattr(xgb_model, "base_model", xgb_model)
cat_base = getattr(cat_boost, "base_model", cat_boost)

# Random Forest feature importance
rf_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_base.feature_importances_
}).sort_values("importance", ascending=False)

print("Random Forest - Top 15 Feature Importances")
display(rf_importance.head(15))

# XGBoost feature importance
xgb_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_base.feature_importances_
}).sort_values("importance", ascending=False)

print("XGBoost - Top 15 Feature Importances")
display(xgb_importance.head(15))

# CatBoost feature importance
cat_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_base.feature_importances_
}).sort_values("importance", ascending=False)

print("CatBoost - Top 15 Feature Importances")
display(cat_importance.head(15))


## Final feature importances para logistic regression


In [ ]:
logreg_base = getattr(logreg_model, "base_model", logreg_model)

logreg_importance = pd.DataFrame({
    "feature": X_test.columns,
    "coefficient": logreg_base.coef_[0],
    "abs_coefficient": np.abs(logreg_base.coef_[0])
}).sort_values("abs_coefficient", ascending=False)

print("Logistic Regression - Top 15 Coefficients by Absolute Value")
display(logreg_importance.head(15))


## SHAP analysis


### SHAP para Logistic Regression


In [ ]:
! uv pip install shap


In [ ]:
import shap

logreg_base = getattr(logreg_model, "base_model", logreg_model)
explainer_logreg = shap.Explainer(logreg_base, X_test)
shap_values_logreg = explainer_logreg(X_test)

shap.plots.beeswarm(shap_values_logreg, max_display=15)


### SHAP para Random Forest


In [ ]:
rf_base = getattr(rf_model, "base_model", rf_model)
explainer_rf = shap.TreeExplainer(rf_base)
shap_values_rf = explainer_rf.shap_values(X_test)

# For binary classification, use class 1 if shap returns a list
if isinstance(shap_values_rf, list):
    shap.summary_plot(shap_values_rf[1], X_test, max_display=15)
else:
    shap.summary_plot(shap_values_rf, X_test, max_display=15)


### SHAP para XGBoost


In [ ]:
xgb_base = getattr(xgb_model, "base_model", xgb_model)
explainer_xgb = shap.TreeExplainer(xgb_base)
shap_values_xgb = explainer_xgb.shap_values(X_test)

shap.summary_plot(shap_values_xgb, X_test, max_display=15)


### SHAP para CatBoost


In [ ]:
cat_base = getattr(cat_boost, "base_model", cat_boost)
explainer_cat = shap.TreeExplainer(cat_base)
shap_values_cat = explainer_cat.shap_values(X_test)

shap.summary_plot(shap_values_cat, X_test, max_display=15)


## Guardar final predictions


In [ ]:
predictions_df = X_test.copy()
predictions_df["y_true"] = y_test.values

for model_name, preds in predictions_dict.items():
    safe_name = model_name.lower().replace(" ", "_")
    predictions_df[f"{safe_name}_pred"] = preds["y_pred"]
    predictions_df[f"{safe_name}_prob"] = preds["y_prob"]

predictions_df.to_csv(Path(r"data/test_set_predictions.csv"), index=False)
final_metrics_df.to_csv(Path(r"data/test_set_final_metrics_frozen_policy.csv"), index=False)

print("Predictions saved to data/test_set_predictions.csv")
print("Final metrics saved to data/test_set_final_metrics_frozen_policy.csv")
display(predictions_df.head())
display(final_metrics_df)
